# 03 — Regime Detection

**STOP POINT**: Inspect HMM regime labels with regime-shaded SPY plot before continuing.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.utils.seeds import set_all_seeds
set_all_seeds()
%matplotlib inline

## 1. Load Required Series

In [ ]:
from src.data.alpaca_loader import load_ticker
from src.data.fred_loader import load_series

spy = load_ticker('SPY', start='2014-01-01')
vix = load_series('VIXCLS', start='2014-01-01')
hy_oas = load_series('BAMLH0A0HYM2', start='2014-01-01')
term_spread = load_series('T10Y2Y', start='2014-01-01')

print(f'SPY: {spy.shape}, VIX: {len(vix)} obs, HY OAS: {len(hy_oas)} obs, Spread: {len(term_spread)} obs')

## 2. Fit Gaussian HMM (Walk-Forward)

In [ ]:
from src.regimes.hmm import fit_hmm_rolling

hmm_probs = fit_hmm_rolling(
    spy_close=spy['close'],
    vix=vix,
    hy_oas=hy_oas,
    term_spread=term_spread,
    initial_train_years=5,
    n_states=4,
    save=True,
)
print(f'HMM probs shape: {hmm_probs.shape}')
hmm_probs.head(10)

## 3. Regime Distribution

In [ ]:
label_counts = hmm_probs['predicted_label'].value_counts()
print('Regime frequency:')
print(label_counts.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
label_counts.plot.bar(ax=ax, color=['#2ca02c','#1f77b4','#ff7f0e','#d62728'], alpha=0.8)
ax.set_title('HMM Regime Frequency')
ax.set_xlabel('Regime')
ax.set_ylabel('Days')
plt.tight_layout()
plt.show()

## 4. Regime-Shaded SPY Plot (STOP POINT)

In [ ]:
from src.evaluation.reporting import plot_regime_shaded_spy
spy_ret = np.log(spy['close'] / spy['close'].shift(1)).dropna()
plot_regime_shaded_spy(spy_ret, hmm_probs['predicted_label'])

# Also display inline
from PIL import Image
img = Image.open('../results/figures/regime_shaded_spy.png')
plt.figure(figsize=(16, 5))
plt.imshow(img)
plt.axis('off')
plt.title('STOP: Inspect regime labels before continuing to modelling')
plt.show()

## 5. LSTM Regime Classifier (after HMM label inspection)

In [ ]:
# Load macro feature panel built by panel.py
from src.utils.io import load_parquet
panel = load_parquet('../data/processed/panel.parquet')

if panel is not None:
    from src.regimes.lstm_regime import run_lstm_regime_walk_forward
    # Use SPY rows only for the macro feature input
    spy_panel = panel.xs('SPY', level='ticker') if 'SPY' in panel.index.get_level_values('ticker') else None
    if spy_panel is not None:
        lstm_probs = run_lstm_regime_walk_forward(
            features_df=spy_panel.drop(columns=['target'], errors='ignore'),
            hmm_probs=hmm_probs,
            initial_train_years=5,
            version='v1',
        )
        print(f'LSTM regime probs: {lstm_probs.shape}')
        print(f'Mean val kappa: {lstm_probs["val_kappa"].mean():.3f}')
else:
    print('Panel not yet built -- run src.data.panel.build_panel() first.')